# Batch summarize Chinese Markdown to English (<=512 tokens) with Ollama

This notebook implements **Method 1**: `Ollama + Qwen + Python` for batch processing.

- Input: Chinese `.md` files (default: `pdf2md/`)
- Output: English `.txt` summaries (<=512 tokens)
- Strategy: chunked map-reduce summarization + final token clipping

In [1]:
# If needed, install once:
# %pip install requests tiktoken tqdm

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import time
import requests
import tiktoken
from tqdm import tqdm

# ===== Config =====
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen2.5:14b-instruct"  # change if your local tag differs

INPUT_DIR = Path(r"d:/MFIN/7036/pdf2md")
OUTPUT_DIR = Path(r"d:/MFIN/7036/md_summary_en_512")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_FINAL_TOKENS = 512
CHUNK_TOKENS = 1800
CHUNK_OVERLAP = 150
REQUEST_TIMEOUT = 600
TEMPERATURE = 0.2

# Batch controls
SKIP_EXISTING = True
MAX_WORKERS = 3  # recommended range: 2-4

enc = tiktoken.get_encoding("cl100k_base")

print("Config loaded.")
print("Input dir:", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Model:", MODEL)
print("SKIP_EXISTING:", SKIP_EXISTING)
print("MAX_WORKERS:", MAX_WORKERS)

Config loaded.
Input dir: d:\MFIN\7036\pdf2md
Output dir: d:\MFIN\7036\md_summary_en_512
Model: qwen2.5:14b-instruct
SKIP_EXISTING: True
MAX_WORKERS: 3


In [2]:
def token_count(text: str) -> int:
    return len(enc.encode(text))

def truncate_to_tokens(text: str, max_tokens: int) -> str:
    ids = enc.encode(text)
    if len(ids) <= max_tokens:
        return text
    return enc.decode(ids[:max_tokens])

def clean_markdown(md: str) -> str:
    md = md.replace('\r\n', '\n')
    md = re.sub(r'```[\s\S]*?```', ' ', md)
    md = re.sub(r'!\[[^\]]*\]\([^)]*\)', ' ', md)
    md = re.sub(r'\[[^\]]*\]\([^)]*\)', ' ', md)
    md = re.sub(r'[#>*`~]{1,}', ' ', md)
    md = re.sub(r'\n{3,}', '\n\n', md)
    md = re.sub(r'\s{2,}', ' ', md)
    return md.strip()

def chunk_text_by_tokens(text: str, chunk_tokens: int = 1800, overlap_tokens: int = 150):
    ids = enc.encode(text)
    chunks = []
    start = 0
    while start < len(ids):
        end = min(start + chunk_tokens, len(ids))
        chunk = enc.decode(ids[start:end])
        chunks.append(chunk)
        if end == len(ids):
            break
        start = max(end - overlap_tokens, start + 1)
    return chunks

In [3]:
def ollama_generate(prompt: str, model: str = MODEL, temperature: float = TEMPERATURE, timeout: int = REQUEST_TIMEOUT):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return data.get("response", "").strip()

MAP_PROMPT = """You are a financial research assistant.
Read the following Chinese markdown segment and produce a concise English summary.
Requirements:
1) Keep factual details (numbers, periods, YoY/QoQ trends, guidance).
2) Focus on key investment-relevant points: business performance, drivers, risks, and outlook.
3) Output plain English prose only.
4) Max 220 English words.

Chinese markdown segment:
{chunk}
"""

REDUCE_PROMPT = """You are a senior analyst.
Combine the following partial English summaries into ONE final summary.
Requirements:
1) Output in English only.
2) Keep only core points, remove repetition.
3) Cover: what happened, why, risks, and forward-looking view.
4) Keep around 280-360 words.
5) Plain text only (no markdown bullets).

Partial summaries:
{partials}
"""

COMPRESS_PROMPT = """Compress the following English summary into a tighter version while preserving all critical facts and directional conclusions.
Output plain English text only, no bullet points, no markdown.

Text:
{text}
"""

def summarize_md_text(md_text: str) -> str:
    cleaned = clean_markdown(md_text)
    if not cleaned:
        return ""

    chunks = chunk_text_by_tokens(cleaned, CHUNK_TOKENS, CHUNK_OVERLAP)
    partials = []

    for chunk in chunks:
        prompt = MAP_PROMPT.format(chunk=chunk)
        out = ollama_generate(prompt)
        if out:
            partials.append(out)

    if not partials:
        return ""

    merged_prompt = REDUCE_PROMPT.format(partials="\n\n".join(partials))
    final_text = ollama_generate(merged_prompt)

    # Enforce <=512 tokens. Try model-side compression first, then hard truncate if still over.
    for _ in range(3):
        if token_count(final_text) <= MAX_FINAL_TOKENS:
            break
        final_text = ollama_generate(COMPRESS_PROMPT.format(text=final_text))

    if token_count(final_text) > MAX_FINAL_TOKENS:
        final_text = truncate_to_tokens(final_text, MAX_FINAL_TOKENS)

    return final_text.strip()

In [4]:
# Quick connectivity test
try:
    pong = ollama_generate("Reply with exactly: OK")
    print("Ollama response:", pong)
except Exception as e:
    print("Connection failed:", e)
    print("Tips: run `ollama serve` and `ollama pull qwen2.5:14b-instruct` first.")

Ollama response: OK


In [5]:
# # Single-file test (recommended before full batch)
# sample_files = sorted(INPUT_DIR.rglob("*.md"))
# print("Found md files:", len(sample_files))

# if sample_files:
#     sample = sample_files[0]
#     print("Testing file:", sample)
#     md_text = sample.read_text(encoding="utf-8", errors="ignore")
#     summary = summarize_md_text(md_text)
#     print("Summary tokens:", token_count(summary))
#     print(summary[:1200])

In [ ]:
# Full batch run (skip existing, process by stock folder, parallel per stock)
md_files = sorted(INPUT_DIR.rglob("*.md"))
print(f"Total md files: {len(md_files)}")

fail_log = []
done_count = 0
skip_count = 0

# Group files by first-level stock folder under INPUT_DIR
stock_to_files = {}
for md_path in md_files:
    rel = md_path.relative_to(INPUT_DIR)
    stock_name = rel.parts[0] if len(rel.parts) > 0 else "_root"
    stock_to_files.setdefault(stock_name, []).append(md_path)

stock_names = sorted(stock_to_files.keys())
total_stocks = len(stock_names)
print(f"Total stock folders: {total_stocks}")

def process_one(md_path):
    rel = md_path.relative_to(INPUT_DIR)
    out_path = OUTPUT_DIR / rel.with_suffix(".txt")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if SKIP_EXISTING and out_path.exists() and out_path.stat().st_size > 0:
        return ("skipped", str(md_path), str(out_path), None, None)

    md_text = md_path.read_text(encoding="utf-8", errors="ignore")
    summary = summarize_md_text(md_text)
    if not summary.strip():
        raise ValueError("Empty summary")

    summary = truncate_to_tokens(summary, MAX_FINAL_TOKENS)
    out_path.write_text(summary, encoding="utf-8")
    return ("done", str(md_path), str(out_path), token_count(summary), None)

overall_pbar = tqdm(total=len(md_files), desc="Overall progress", position=0)
stock_pbar = None

for stock_idx, stock_name in enumerate(stock_names, start=1):
    stock_files = stock_to_files[stock_name]
    stock_total = len(stock_files)

    print()
    print(f"Converted {stock_idx - 1}/{total_stocks}. Now converting stock {stock_idx}/{total_stocks}: {stock_name}")

    if stock_pbar is not None:
        stock_pbar.close()
    stock_pbar = tqdm(total=stock_total, desc="Current stock progress", position=1, leave=False)

    completed_in_stock = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one, md_path) for md_path in stock_files]
        for fut in as_completed(futures):
            try:
                status, md_path_str, out_path_str, summary_tokens, _ = fut.result()
                if status == "done":
                    done_count += 1
                else:
                    skip_count += 1
            except Exception as e:
                fail_log.append(str(e))
            finally:
                completed_in_stock += 1
                stock_pbar.set_description(f"Current stock progress: {completed_in_stock}/{stock_total}")
                stock_pbar.update(1)
                overall_pbar.update(1)

if stock_pbar is not None:
    stock_pbar.close()
overall_pbar.close()

print("Done.")
print("Generated:", done_count)
print("Skipped existing:", skip_count)
print("Failures:", len(fail_log))
if fail_log:
    for item in fail_log[:20]:
        print(item)

Total md files: 41224
Total stock folders: 302


Overall progress:   0%|          | 0/41224 [00:00<?, ?it/s]


Converted 0/302. Now converting stock 1/302: TCL中环


Overall progress:   0%|          | 76/41224 [00:00<00:54, 759.39it/s]


Converted 1/302. Now converting stock 2/302: TCL科技


...

Overall progress:  53%|█████▎    | 21821/41224 [17:40:52<159:43:53, 29.64s/it]


Converted 177/302. Now converting stock 178/302: 恒力石化


Overall progress:  53%|█████▎    | 21995/41224 [19:42:14<142:19:18, 26.65s/it] 


Converted 178/302. Now converting stock 179/302: 恒瑞医药


Overall progress:  54%|█████▍    | 22168/41224 [21:37:54<179:45:04, 33.96s/it]

In [ ]:
# Optional: save failure log
if 'fail_log' in globals() and fail_log:
    log_path = OUTPUT_DIR / "_failed_files.json"
    log_path.write_text(json.dumps(fail_log, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Failure log saved to:", log_path)
else:
    print("No failures.")